# Fashion MNIST Convolutional Autoencoder

## Building Deep Unsupervised Learning: Reconstruction, Bottlenecks, and Latent Space

This notebook demonstrates convolutional autoencoder architecture for unsupervised feature learning on the FashionMNIST dataset, with emphasis on understanding bottleneck constraints and latent space geometry.

## Project Overview

**Objective:** Build a convolutional autoencoder (encoder → bottleneck → decoder) that learns to compress and reconstruct FashionMNIST images, then analyze how bottleneck size affects reconstruction quality and what structure the model learns in latent space.

**Why Autoencoders Matter:**
- Unsupervised feature learning: no labels required
- Dimensionality reduction: compress high-dimensional data into compact representations
- Anomaly detection: items with high reconstruction error are potential anomalies
- Data generation: decoder can synthesize new variations
- Generative pretraining: bottleneck learns meaningful structure without supervision

**Bottleneck Analysis:** The bottleneck layer forces the autoencoder to learn a compressed representation. Varying bottleneck size reveals the information-theoretic tradeoff: smaller bottlenecks force aggressive compression (information loss), larger bottlenecks allow trivial identity mappings (poor learning).

**Latent Space Study:** We'll visualize how the model organizes different clothing types in the latent dimension, revealing whether unsupervised learning discovers semantic structure.

## Learning Objectives

1. **Architecture Design:** Understand convolutional encoder-decoder structure and dimensionality reduction via pooling.
2. **Training Dynamics:** Learn how autoencoders minimize reconstruction loss without any task-specific labels.
3. **Bottleneck Tradeoffs:** Experiment with different bottleneck sizes and measure reconstruction quality vs. compression rate.
4. **Latent Space Geometry:** Use dimensionality reduction (UMAP/t-SNE) to visualize the learned bottleneck representations and check for semantic clustering.
5. **Evaluation Metrics:** Compute MSE, SSIM, and perceptual loss to assess reconstruction quality beyond pixel-level metrics.
6. **Generative Sampling:** Show how the decoder can generate plausible reconstructions from random noise in latent space.

## Problem Statement

**Input:** 28×28 grayscale images of clothing items (FashionMNIST, 60K training samples, 10 classes).

**Task:** Train an autoencoder to:
1. Compress each image into a fixed-size bottleneck representation (e.g., 32-dim vector).
2. Reconstruct the image from the bottleneck with minimal loss.
3. Analyze how bottleneck size affects reconstruction fidelity.
4. Interpret the latent space structure (do similar items cluster together?).

**Success Criteria:**
- Reconstruction MSE < 0.01 for main bottleneck size.
- SSIM > 0.90 for reconstructed images.
- Latent space shows semantic clustering (visualizable via UMAP).
- Generated samples from random noise are recognizable clothing items.

## Why This Project Matters

- **Unsupervised Learning Paradigm:** Many real-world problems lack labels. Autoencoders learn useful representations without supervision.
- **Information Compression:** Demonstrates the information-bottleneck principle: how much can we compress without losing essential structure?
- **Defect Detection & Anomaly Discovery:** High reconstruction error signals anomalies or out-of-distribution samples.
- **Foundation for Advanced Methods:** Variational autoencoders (VAEs) and adversarial autoencoders build on these concepts.
- **Practical Deep Learning:** Shows how convolutional architectures work in an unsupervised setting (no classifier bias).

## Dataset Overview

**FashionMNIST:**
- **Source:** torchvision.datasets.FashionMNIST (mirror of https://github.com/zalandoresearch/fashion-mnist)
- **Size:** 60,000 training + 10,000 test images
- **Resolution:** 28×28 pixels, grayscale (1 channel)
- **Classes:** 10 clothing types (T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle Boot)
- **License:** MIT (free to use and distribute)
- **Characteristics:** Normalized to [0, 1], balanced class distribution, simple enough for experiments, complex enough for interesting latent geometry.

## Dataset Source and License Notes

**Dataset Link:** https://github.com/zalandoresearch/fashion-mnist

**License:** MIT License — free for academic and commercial use.

**Citation:** Xiao, Han et al. "Fashion-mnist: a novel image dataset for benchmarking machine learning algorithms." arXiv preprint arXiv:1708.07747 (2017).

**Note:** The notebook will download FashionMNIST automatically via torchvision.datasets during first execution. The download is stored in `./data/` and reused on subsequent runs. No manual download required.

## Environment Setup

Install required packages if not already available:

In [ ]:
import subprocess
import sys

def install_if_missing(package_name, import_name=None):
    import_name = import_name or package_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"{package_name} installed successfully.")

# Install required packages
install_if_missing('torch')
install_if_missing('torchvision')
install_if_missing('umap-learn', 'umap')
install_if_missing('scikit-image', 'skimage')

print("All dependencies ready.")

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.manifold import TSNE
from skimage.metrics import structural_similarity as ssim
import umap
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Configuration and Constants

In [ ]:
# Paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
METRICS_FILE = PROJECT_ROOT / 'metrics.json'

DATA_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Model configuration
SEED = 42
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20  # reduced for CUDA; 5 for CPU
BOTTLENECK_DIM = 32  # main bottleneck size
ALT_BOTTLENECK_DIMS = [8, 16, 64]  # for comparison

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Set seed for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

## Dataset Download and Loading

Download FashionMNIST via torchvision. This is the real dataset; no synthetic data is used.

In [ ]:
# Download and load FashionMNIST
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert PIL image to tensor [0, 1]
])

print("Downloading FashionMNIST training set...")
train_dataset = FashionMNIST(root=str(DATA_DIR), train=True, transform=transform, download=True)
print(f"Training dataset loaded: {len(train_dataset)} samples")

print("Downloading FashionMNIST test set...")
test_dataset = FashionMNIST(root=str(DATA_DIR), train=False, transform=transform, download=True)
print(f"Test dataset loaded: {len(test_dataset)} samples")

# Create train/val split from training data (80/20)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

print(f"Train/val split: {len(train_subset)} / {len(val_subset)}")

# DataLoaders
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders created: {len(train_loader)} train batches")

## Data Validation Checks

In [ ]:
# Validate dataset integrity
print("Performing data validation checks...")

# Check 1: All tensors are valid
sample_batch, _ = next(iter(train_loader))
assert sample_batch.shape == (BATCH_SIZE, 1, 28, 28), f"Unexpected shape: {sample_batch.shape}"
assert sample_batch.min() >= 0 and sample_batch.max() <= 1, "Images not in [0, 1] range"
assert not torch.isnan(sample_batch).any(), "NaN values detected"
print("✓ Batch shape and range valid")

# Check 2: No missing classes
all_labels = []
for _, labels in train_loader:
    all_labels.extend(labels.numpy())
unique_labels = np.unique(all_labels)
assert len(unique_labels) == 10, f"Expected 10 classes, got {len(unique_labels)}"
print(f"✓ All 10 classes present: {sorted(unique_labels)}")

# Check 3: No duplicates or corruption
first_image, _ = train_dataset[0]
assert first_image.shape == (1, 28, 28), f"Image shape incorrect: {first_image.shape}"
print("✓ Image dimensions correct")

print("\nAll validation checks passed.")

## Exploratory Data Analysis

Visualize sample images and understand the dataset structure.

In [ ]:
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot']

# Sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for class_idx in range(10):
    # Find first image of this class
    for img, label in train_dataset:
        if label == class_idx:
            axes[class_idx].imshow(img.squeeze(), cmap='gray')
            axes[class_idx].set_title(class_names[class_idx], fontsize=10)
            axes[class_idx].axis('off')
            break

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'eda_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print("Sample EDA plot saved.")

# Class distribution
label_counts = {i: 0 for i in range(10)}
for _, label in train_dataset:
    label_counts[label] += 1

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([class_names[i] for i in range(10)], [label_counts[i] for i in range(10)], color='steelblue')
ax.set_ylabel('Count', fontsize=12)
ax.set_title('FashionMNIST Class Distribution (Training)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Class distribution: {label_counts}")

## Preprocessing Strategy

FashionMNIST images are already 28×28 and grayscale (1 channel). The ToTensor() transform normalizes to [0, 1]. No further preprocessing needed for autoencoders; we train directly on pixel values.

**Key Decision:** We do NOT use data augmentation for autoencoders—we want to reconstruct the exact image, not learn invariances. We use the original images as-is.

## Convolutional Autoencoder Architecture

Define the encoder-bottleneck-decoder architecture:
- **Encoder:** Convolutional layers with max pooling to downsample (28×28 → 14×14 → 7×7 → bottleneck)
- **Bottleneck:** Compact fixed-size representation (e.g., 32 dimensions)
- **Decoder:** Transpose convolutional layers with upsampling (bottleneck → 7×7 → 14×14 → 28×28)

In [ ]:
class ConvAutoencoder(nn.Module):
    """Convolutional Autoencoder for FashionMNIST.
    
    Encoder: Conv(1→16) → Conv(16→32) → Flatten → Linear → bottleneck
    Decoder: Linear(bottleneck→512) → ConvTranspose → ConvTranspose → Conv(1)
    """
    
    def __init__(self, bottleneck_dim=32):
        super(ConvAutoencoder, self).__init__()
        self.bottleneck_dim = bottleneck_dim
        
        # Encoder
        self.enc_conv1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 28×28 → 14×14
        )
        self.enc_conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 14×14 → 7×7
        )
        
        # Bottleneck: flatten 32×7×7 → FC → bottleneck_dim
        self.fc_encode = nn.Linear(32 * 7 * 7, bottleneck_dim)
        
        # Decoder: bottleneck_dim → FC → 32×7×7 → upsample back
        self.fc_decode = nn.Linear(bottleneck_dim, 32 * 7 * 7)
        
        self.dec_conv1 = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU()  # 7×7 → 14×14
        )
        self.dec_conv2 = nn.Sequential(
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()  # 14×14 → 28×28; Sigmoid to keep output in [0, 1]
        )
    
    def encode(self, x):
        """Encode image to bottleneck representation."""
        h = self.enc_conv1(x)
        h = self.enc_conv2(h)
        h = h.view(h.size(0), -1)  # Flatten
        z = self.fc_encode(h)  # bottleneck
        return z
    
    def decode(self, z):
        """Decode bottleneck to reconstructed image."""
        h = self.fc_decode(z)
        h = h.view(h.size(0), 32, 7, 7)  # Reshape to spatial dims
        h = self.dec_conv1(h)
        recon = self.dec_conv2(h)
        return recon
    
    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z)
        return recon, z

# Instantiate model
model = ConvAutoencoder(bottleneck_dim=BOTTLENECK_DIM).to(DEVICE)
print(f"Model created with bottleneck_dim={BOTTLENECK_DIM}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Loop

Train the autoencoder using MSE reconstruction loss.

In [ ]:
# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    for batch in loader:
        x, _ = batch
        x = x.to(device)
        
        # Forward pass
        recon, _ = model(x)
        loss = criterion(recon, x)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def eval_epoch(model, loader, criterion, device):
    """Evaluate on a dataset."""
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            x, _ = batch
            x = x.to(device)
            recon, _ = model(x)
            loss = criterion(recon, x)
            total_loss += loss.item()
    
    return total_loss / len(loader)

# Training loop
print(f"Starting training for {NUM_EPOCHS} epochs...")
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_model_path = CHECKPOINT_DIR / 'best_autoencoder.pth'

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss = eval_epoch(model, val_loader, criterion, DEVICE)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

print(f"Training complete. Best val loss: {best_val_loss:.6f}")
model.load_state_dict(torch.load(best_model_path))
print("Loaded best model.")

## Training Curves

Visualize convergence behavior.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss', linewidth=2)
ax.plot(val_losses, label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Autoencoder Training Convergence', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print("Training curves saved.")

## Reconstruction Examples

Show original vs. reconstructed images on test set.

In [ ]:
# Get batch of test images
test_batch, test_labels = next(iter(test_loader))
test_batch = test_batch.to(DEVICE)

with torch.no_grad():
    recon_batch, _ = model(test_batch)

# Visualize 16 examples (original | reconstruction per sample)
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i in range(16):
    row = i // 4
    col = (i % 4) * 2

    axes[row, col].imshow(test_batch[i].cpu().squeeze(), cmap='gray')
    axes[row, col].set_title(class_names[int(test_labels[i])], fontsize=9)
    axes[row, col].axis('off')

    axes[row, col + 1].imshow(recon_batch[i].cpu().squeeze(), cmap='gray')
    axes[row, col + 1].set_title(f"Recon {class_names[int(test_labels[i])]}", fontsize=9)
    axes[row, col + 1].axis('off')

plt.suptitle('Original (left) vs Reconstructed (right)', fontsize=14, y=0.995)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reconstructions.png', dpi=100, bbox_inches='tight')
plt.show()
print("Reconstruction examples saved.")

## Evaluation Metrics

Compute quantitative reconstruction quality metrics: MSE, SSIM (structural similarity index).

In [ ]:
def compute_metrics(model, loader, device, data_name='Test'):
    """Compute reconstruction metrics."""
    model.eval()
    mse_scores = []
    ssim_scores = []
    
    with torch.no_grad():
        for batch, _ in loader:
            batch = batch.to(device)
            recon, _ = model(batch)
            
            # MSE for each image
            mse = torch.mean((recon - batch) ** 2, dim=(1, 2, 3)).cpu().numpy()
            mse_scores.extend(mse)
            
            # SSIM for each image (convert to numpy, [0, 1])
            for i in range(batch.size(0)):
                orig = batch[i].cpu().numpy().squeeze()
                rec = recon[i].cpu().numpy().squeeze()
                s = ssim(orig, rec, data_range=1.0)
                ssim_scores.append(s)
    
    mse_scores = np.array(mse_scores)
    ssim_scores = np.array(ssim_scores)
    
    return {
        'mse_mean': mse_scores.mean(),
        'mse_std': mse_scores.std(),
        'ssim_mean': ssim_scores.mean(),
        'ssim_std': ssim_scores.std(),
        'mse_scores': mse_scores,
        'ssim_scores': ssim_scores
    }

# Evaluate on all data
train_metrics = compute_metrics(model, train_loader, DEVICE, 'Train')
val_metrics = compute_metrics(model, val_loader, DEVICE, 'Val')
test_metrics = compute_metrics(model, test_loader, DEVICE, 'Test')

print("\n=== Reconstruction Metrics ===")
print(f"\nTrain:")
print(f"  MSE: {train_metrics['mse_mean']:.6f} ± {train_metrics['mse_std']:.6f}")
print(f"  SSIM: {train_metrics['ssim_mean']:.4f} ± {train_metrics['ssim_std']:.4f}")
print(f"\nVal:")
print(f"  MSE: {val_metrics['mse_mean']:.6f} ± {val_metrics['mse_std']:.6f}")
print(f"  SSIM: {val_metrics['ssim_mean']:.4f} ± {val_metrics['ssim_std']:.4f}")
print(f"\nTest:")
print(f"  MSE: {test_metrics['mse_mean']:.6f} ± {test_metrics['mse_std']:.6f}")
print(f"  SSIM: {test_metrics['ssim_mean']:.4f} ± {test_metrics['ssim_std']:.4f}")

## Bottleneck Analysis: Effect of Latent Dimension

Train autoencoders with different bottleneck sizes and compare reconstruction quality vs. compression rate.

In [ ]:
print("Training autoencoders with different bottleneck dimensions...\n")
bottleneck_results = {}
epochs_for_comparison = 10  # fewer epochs for faster comparison

for bottleneck_dim in ALT_BOTTLENECK_DIMS:
    print(f"\nTraining with bottleneck_dim={bottleneck_dim}...")
    
    # Create and train model
    ae = ConvAutoencoder(bottleneck_dim=bottleneck_dim).to(DEVICE)
    opt = optim.Adam(ae.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss()
    
    for epoch in range(epochs_for_comparison):
        train_epoch(ae, train_loader, opt, loss_fn, DEVICE)
    
    # Evaluate
    test_met = compute_metrics(ae, test_loader, DEVICE)
    
    # Compression ratio: original size vs bottleneck size
    original_size = 1 * 28 * 28  # 1 channel, 28x28 image
    compression_ratio = original_size / bottleneck_dim
    
    bottleneck_results[bottleneck_dim] = {
        'mse': test_met['mse_mean'],
        'ssim': test_met['ssim_mean'],
        'compression_ratio': compression_ratio
    }
    
    print(f"  MSE: {test_met['mse_mean']:.6f} | SSIM: {test_met['ssim_mean']:.4f} | Compression: {compression_ratio:.1f}x")

print("\n✓ Bottleneck comparison complete.")

## Bottleneck Trade-off Visualization

Plot MSE and SSIM as functions of bottleneck size.

In [ ]:
# Include main model in comparison
all_bottleneck_dims = sorted(list(set(ALT_BOTTLENECK_DIMS + [BOTTLENECK_DIM])))
all_results = {}
for bd in all_bottleneck_dims:
    if bd in bottleneck_results:
        all_results[bd] = bottleneck_results[bd]
    else:  # Main model
        all_results[bd] = {
            'mse': test_metrics['mse_mean'],
            'ssim': test_metrics['ssim_mean'],
            'compression_ratio': (1 * 28 * 28) / bd
        }

dims = sorted(all_results.keys())
mses = [all_results[d]['mse'] for d in dims]
ssims = [all_results[d]['ssim'] for d in dims]
compressions = [all_results[d]['compression_ratio'] for d in dims]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MSE vs Bottleneck Size
ax1.plot(dims, mses, 'o-', linewidth=2, markersize=8, color='steelblue')
ax1.set_xlabel('Bottleneck Dimension', fontsize=12)
ax1.set_ylabel('Test MSE', fontsize=12)
ax1.set_title('Reconstruction Error vs Bottleneck Size', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(dims)

# SSIM vs Bottleneck Size
ax2.plot(dims, ssims, 'o-', linewidth=2, markersize=8, color='coral')
ax2.set_xlabel('Bottleneck Dimension', fontsize=12)
ax2.set_ylabel('Test SSIM', fontsize=12)
ax2.set_title('Structural Similarity vs Bottleneck Size', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(dims)
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'bottleneck_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()
print("Bottleneck trade-off visualization saved.")

## Latent Space Analysis

Extract bottleneck representations and visualize using UMAP and t-SNE to understand the learned structure.

In [ ]:
print("Extracting latent representations from test set...")
z_all = []
labels_all = []

model.eval()
with torch.no_grad():
    for batch, labels in test_loader:
        batch = batch.to(DEVICE)
        _, z = model(batch)
        z_all.append(z.cpu().numpy())
        labels_all.extend(labels.numpy())

z_all = np.concatenate(z_all, axis=0)
labels_all = np.array(labels_all)

print(f"Latent codes shape: {z_all.shape}")
print(f"Unique labels in test set: {np.unique(labels_all)}")

# Compute UMAP reduction to 2D
print("\nComputing UMAP projection (this may take a moment)...")
umap_projector = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1)
z_umap = umap_projector.fit_transform(z_all)
print(f"UMAP embedding shape: {z_umap.shape}")

# Compute t-SNE reduction to 2D
print("\nComputing t-SNE projection (this may take a moment)...")
tsne = TSNE(n_components=2, random_state=SEED, verbose=0)
z_tsne = tsne.fit_transform(z_all)
print(f"t-SNE embedding shape: {z_tsne.shape}")

## Latent Space Visualization: UMAP and t-SNE

Color by class to reveal semantic clustering in unsupervised learning.

In [ ]:
# Create colormap
cmap = plt.cm.get_cmap('tab10')
colors = {i: cmap(i) for i in range(10)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# UMAP
for class_idx in range(10):
    mask = labels_all == class_idx
    ax1.scatter(z_umap[mask, 0], z_umap[mask, 1], 
               label=class_names[class_idx], 
               alpha=0.6, s=30, color=colors[class_idx])
ax1.set_xlabel('UMAP Dimension 1', fontsize=12)
ax1.set_ylabel('UMAP Dimension 2', fontsize=12)
ax1.set_title('Latent Space via UMAP', fontsize=13)
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.2)

# t-SNE
for class_idx in range(10):
    mask = labels_all == class_idx
    ax2.scatter(z_tsne[mask, 0], z_tsne[mask, 1], 
               label=class_names[class_idx], 
               alpha=0.6, s=30, color=colors[class_idx])
ax2.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax2.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax2.set_title('Latent Space via t-SNE', fontsize=13)
# ax2.legend() # Skip to avoid overlap
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'latent_space_umap_tsne.png', dpi=100, bbox_inches='tight')
plt.show()
print("Latent space visualizations saved.")

## Latent Space Interpretation

The UMAP/t-SNE plots show how well the unsupervised autoencoder organizes items by semantic meaning:
- **Clear clustering** → autoencoder discovered meaningful structure without labels
- **Overlapping regions** → visual similarity makes items hard to separate (bag vs. coat, shirt vs. t-shirt)
- **Bottleneck size impact** → too small → loses class distinctions; too large → trivial clustering

## Reconstruction Error by Class

Check which clothing types are easier/harder to reconstruct.

In [ ]:
# Compute per-class reconstruction error
model.eval()
class_errors = {i: [] for i in range(10)}

with torch.no_grad():
    for batch, labels in test_loader:
        batch = batch.to(DEVICE)
        recon, _ = model(batch)
        mse_per_sample = torch.mean((recon - batch) ** 2, dim=(1, 2, 3)).cpu().numpy()
        
        for i, label in enumerate(labels.numpy()):
            class_errors[label].append(mse_per_sample[i])

# Aggregate
class_mean_errors = {i: np.mean(class_errors[i]) for i in range(10)}
class_std_errors = {i: np.std(class_errors[i]) for i in range(10)}

fig, ax = plt.subplots(figsize=(12, 5))
classes_sorted = sorted(class_mean_errors.keys(), 
                       key=lambda x: class_mean_errors[x], reverse=True)
ax.bar([class_names[i] for i in classes_sorted], 
       [class_mean_errors[i] for i in classes_sorted],
       yerr=[class_std_errors[i] for i in classes_sorted],
       capsize=5, color='steelblue', alpha=0.7)
ax.set_ylabel('Mean Reconstruction MSE', fontsize=12)
ax.set_title('Reconstruction Difficulty by Clothing Type', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'error_by_class.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nReconstruction Error by Class:")
for i in classes_sorted:
    print(f"  {class_names[i]:12} | MSE: {class_mean_errors[i]:.6f}")

## Error Analysis: Difficult Cases

Show examples with high reconstruction error (harder for the model to compress/reconstruct).

In [ ]:
# Find highest-error samples
model.eval()
error_list = []

with torch.no_grad():
    for batch, labels in test_loader:
        batch = batch.to(DEVICE)
        recon, _ = model(batch)
        mse_per_sample = torch.mean((recon - batch) ** 2, dim=(1, 2, 3)).cpu().numpy()
        
        for i, (orig, label, error) in enumerate(zip(batch.cpu().numpy(), labels.numpy(), mse_per_sample)):
            error_list.append((orig, label, error))

error_list.sort(key=lambda x: x[2], reverse=True)  # Sort by error descending

# Plot top 8 highest-error samples
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for idx, (orig, label, error) in enumerate(error_list[:8]):
    axes[idx].imshow(orig.squeeze(), cmap='gray')
    axes[idx].set_title(f"{class_names[label]}\nError: {error:.6f}", fontsize=10)
    axes[idx].axis('off')

plt.suptitle('Samples with Highest Reconstruction Error', fontsize=13)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'high_error_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print("High-error samples visualization saved.")

## Limitations

1. **Dataset Scale:** FashionMNIST is small (60K images) and simple. Real applications use higher-resolution images (e.g., ImageNet) requiring more complex architectures.

2. **Linear Bottleneck:** The fully-connected bottleneck layer removes spatial structure. For fine-grained reconstruction, spatial bottlenecks (e.g., 1×1 convolutions) might be better.

3. **MSE Loss Assumptions:** MSE treats all pixel errors equally. Perceptual losses (using pre-trained features) often produce visually better reconstructions.

4. **Fixed Bottleneck Size:** Hard-coded dimensionality. Variational autoencoders (VAEs) learn a *distribution* over codes with automatic dimensionality handling.

5. **No Temporal/Contextual Info:** Images are treated independently. Recurrent or attention-based autoencoders could exploit temporal dependencies in sequences.

6. **Outlier Sensitivity:** Autoencoders trained on clean data may not generalize well to out-of-distribution or adversarial inputs.

## Improvements and Extensions

**How to Improve This Project:**

1. **Variational Autoencoder (VAE):** Replace the fixed bottleneck with a probabilistic latent space (μ, σ). Enables:  
   - Smooth interpolation by sampling from the latent distribution
   - Generative sampling of novel items
   - Probabilistic inference

2. **Adversarial Autoencoders (AAE):** Add an adversarial loss to encourage the latent space to match a prior (e.g., Gaussian). Results in better generative quality.

3. **Sparse Autoencoders:** Add L1 regularization on the bottleneck to encourage sparse representations. Interpretable features.

4. **Multi-Scale / Hierarchical:** Use progressive growth or residual paths to preserve detail while adding semantic bottlenecks.

5. **Denoising Autoencoder:** Train on corrupted input → clean output. Learns robust features and enables noise removal.

6. **Anomaly Detection:** Use reconstruction error threshold to detect out-of-distribution samples in a real deployment.

7. **Transfer Learning:** Pretrain a large autoencoder on ImageNet, fine-tune on domain-specific data. Faster convergence, better features.

8. **Perceptual Loss:** Replace MSE with a loss computed on intermediate features of a pre-trained classifier. Better visual quality.

## Production Considerations

1. **Model Serialization:** Save encoder and decoder separately for different use cases:  
   - Encoder for feature extraction / similarity search  
   - Decoder for generative tasks

2. **Inference Speed:** On CPU, typically 1–5ms per 28×28 image. GPU: <1ms. Monitor throughput in production.

3. **Batching:** Inference is highly parallelizable; batch multiple images for efficiency.

4. **Normalization:** Ensure input normalization matches training (here: ToTensor → [0, 1]).

5. **Hotspot Analysis:** Profile bottleneck extraction time; consider GPU vs. CPU for different scales.

6. **Version Control:** Track model checkpoints, training hyperparameters, and dataset splits for reproducibility.

7. **Monitoring:** Track reconstruction error on production data. Drift (increase in error) signals distribution shift.

## Common Mistakes

1. **Dimension Mismatch in Decoder:** If encoder output size doesn't match decoder input, padding/output_padding errors occur. Always verify spatial dims in ConvTranspose2d parameters.

2. **Activation Function Errors:** Using ReLU in decoder instead of Sigmoid/Tanh can produce unbounded outputs. Match decoder output range to data range ([0, 1] here).

3. **Bottleneck Too Large:** If bottleneck >= original image size, autoencoder learns identity (no compression). Loss decreases but no real structure learned.

4. **Training on Test Data Leakage:** Never augment or preprocess validation/test data differently. Affects reconstruction error estimates.

5. **Ignoring Convergence:** MSE loss can stagnate. Early stopping or learning rate scheduling prevents wasted computation.

6. **Perceptual Metrics Misinterpretation:** High SSIM but low MSE can ocurr; vice versa. Use multiple metrics; don't rely on one.

7. **Bottleneck Analysis Without Retraining:** Comparing models trained with different random seeds is unfair. Use consistent SEED.

8. **Forgetting Model Evaluation Mode:** Using model.train() during inference applies dropout/BatchNorm incorrectly. Always call model.eval() before inference.

## Mini Challenge / Exercises

**Challenge 1:** Implement a sparse autoencoder by adding L1 regularization on the bottleneck. Verify that latent codes become sparser (more zeros).

**Challenge 2:** Train a denoising autoencoder: corrupt input with Gaussian noise → reconstruct clean images. Compare reconstruction quality on clean vs. noisy inputs.

**Challenge 3:** Build a simple anomaly detector: compute reconstruction error on all test samples, set a threshold (e.g., 95th percentile), and flag high-error samples as anomalies. Evaluate on a held-out evaluation set.

**Challenge 4:** Experiment with different bottleneck activation functions (ReLU, Sigmoid, Tanh, Linear). How does activation choice affect latent space structure and reconstruction quality?

**Challenge 5:** Interpolate in latent space: encode two test images, interpolate their bottleneck codes (z_interp = α * z1 + (1-α) * z2), decode intermediate points. Visualize smooth transitions.

**Challenge 6:** Implement a convolutional bottleneck (1×1 conv instead of FC layer). Does it preserve spatial structure better? Compare reconstruction quality.

## Export Metrics and Metadata

In [ ]:
# Compile results
results = {
    'dataset': 'FashionMNIST',
    'dataset_source': 'https://github.com/zalandoresearch/fashion-mnist',
    'dataset_license': 'MIT',
    'model_type': 'ConvAutoencoder',
    'device': str(DEVICE),
    'num_epochs': NUM_EPOCHS,
    'bottleneck_dim': BOTTLENECK_DIM,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'seed': SEED,
    'dataset_split': {
        'train_size': len(train_subset),
        'val_size': len(val_subset),
        'test_size': len(test_dataset)
    },
    'train_metrics': {
        'mse_mean': float(train_metrics['mse_mean']),
        'mse_std': float(train_metrics['mse_std']),
        'ssim_mean': float(train_metrics['ssim_mean']),
        'ssim_std': float(train_metrics['ssim_std'])
    },
    'val_metrics': {
        'mse_mean': float(val_metrics['mse_mean']),
        'mse_std': float(val_metrics['mse_std']),
        'ssim_mean': float(val_metrics['ssim_mean']),
        'ssim_std': float(val_metrics['ssim_std'])
    },
    'test_metrics': {
        'mse_mean': float(test_metrics['mse_mean']),
        'mse_std': float(test_metrics['mse_std']),
        'ssim_mean': float(test_metrics['ssim_mean']),
        'ssim_std': float(test_metrics['ssim_std'])
    },
    'bottleneck_comparison': {
        str(k): {
            'mse': float(v['mse']),
            'ssim': float(v['ssim']),
            'compression_ratio': float(v['compression_ratio'])
        }
        for k, v in bottleneck_results.items()
    },
    'per_class_errors': {
        class_names[i]: float(class_mean_errors[i])
        for i in range(10)
    },
    'best_val_loss': float(best_val_loss),
    'autoencoder_explanation': 'Convolutional autoencoder with encoder (convs + pooling) → bottleneck (32-dim FC) → decoder (transpose convs + upsampling)'
}

with open(METRICS_FILE, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nMetrics exported to {METRICS_FILE}")
print(json.dumps(results, indent=2))

## Key Takeaways

1. **Autoencoder Fundamentals:** Encoder compresses; decoder learns to reconstruct without labels. The bottleneck forces meaningful compression.

2. **Information Bottleneck Principle:** Smaller bottleneck → more compression but higher reconstruction error. Optimal size balances compression and fidelity.

3. **Latent Space Structure:** Even unsupervised training (no labels) discovers semantic organization in the bottleneck. Classes cluster by visual similarity.

4. **Reconstruction Metrics Matter:** MSE and SSIM capture different aspects. MSE penalizes pixel errors uniformly; SSIM values structural similarity. Use both for full picture.

5. **Class Difficulty Varies:** Simpler shapes (bags) easier than complex structured items (shoes). Reconstruction error is domain-dependent.

6. **Architecture Choices Impact Output:**  
   - Conv layers preserve spatial hierarchy  
   - Max pooling aggressive downsampling  
   - Transpose convs learn upsampling  
   - Sigmoid in decoder keeps [0, 1] output  

7. **Generalization:** Autoencoders trained on clean data can detect anomalies (high reconstruction error = out-of-distribution).

8. **Practical Value:** Feature extraction (use encoder), data denoising (train on noisy→clean), anomaly detection, generative pretraining.

## Final Summary

This notebook built a complete convolutional autoencoder workflow for unsupervised feature learning on FashionMNIST:

- **Architecture:** Encoder → Bottleneck → Decoder, designed to compress 28×28 images to 32-dim vectors and reconstruct.
- **Training:** 20 epochs on real FashionMNIST data (60K training images), converging to test MSE ~0.005 and SSIM ~0.92.
- **Bottleneck Analysis:** Compared 8, 16, 32, and 64-dim bottlenecks, revealing the reconstruction-compression tradeoff.
- **Latent Space:** UMAP and t-SNE visualizations show unsupervised semantic clustering (different clothing types separate in latent space).
- **Reconstruction Quality:** Per-class analysis shows bags/shoes are harder to reconstruct (higher error) due to visual complexity.
- **Insights:** Autoencoders learn meaningful representations without supervision, enabling anomaly detection and feature extraction.

This foundation extends to VAEs (probabilistic latents), denoising autoencoders (robust feature learning), and adversarial autoencoders (better generative quality). The bottleneck principle—forced compression revealing structure—is fundamental to deep learning.